# Phần của Hiệp — Logistic Regression và HistGradientBoosting
Notebook chạy từ bản clone repo, kế thừa src/preprocessing.py và src/evaluation.py.
Không cần chạy lại EDA hoặc tạo lại manifest. Cài requirements.txt trước khi chạy.

Quy trình: CV train → chốt cấu hình/mô hình → ngưỡng validation → test một lần.
Chạy ô huấn luyện sẽ mất vài phút, ghi lại bảng số liệu và báo cáo của phần Hiệp.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src/modeling.py').exists():
    root = root.parent
if not (root / 'src/modeling.py').exists():
    raise RuntimeError('Mở notebook trong bản clone repo, ở thư mục gốc hoặc notebooks/.')
sys.path.insert(0, str(root))
from IPython.display import display, Image, Markdown
from src.modeling import load_data, model_specs, run
from src.evaluation import split_table


## 1. Kiểm tra dữ liệu và phép chia chuẩn

In [ ]:
parts, groups = load_data()
display(split_table(parts))
for name, (pipeline, grid) in model_specs().items():
    print(name, grid)
    display(pipeline)


## 2. Huấn luyện và đánh giá
Lưới đã định trước: LR có 3 cấu hình, HGB có 4 cấu hình, 5 fold theo nhóm.
Mọi phép học chuẩn hóa nằm trong pipeline. Validation chỉ chọn ngưỡng.
Không sửa cấu hình theo kết quả test.


In [ ]:
result = run()
display(result['cv'])
print('Mô hình được chọn bằng CV train:', result['selected'])
display(result['test'])
display(result['comparisons'])


## 3. Biểu đồ và báo cáo

In [ ]:
display(Image(filename=str(root / 'reports/figures/11_pr_roc.png')))
display(Image(filename=str(root / 'reports/figures/11_confusion.png')))
display(Markdown((root / 'reports/Bao_cao_Mo_hinh_Hiep.md').read_text(encoding='utf-8')))


## 4. Nạp pipeline cho demo
Dữ liệu đầu vào phải là các cột FEATURES đã được làm sạch theo quy tắc Phần 2.
Pipeline tự tạo thêm biến và chuẩn hóa; không truyền CSV đã chuẩn hóa vào đây.


In [ ]:
import joblib
from src.preprocessing import FEATURES
bundle = joblib.load(root / 'outputs/hiep/logistic.joblib')
example = parts['validation'][FEATURES].head(5)
p = bundle['model'].predict_proba(example)[:, 1]
print('Xác suất:', p)
print('Cảnh báo:', p >= bundle['threshold'])
